# Day 25 · DPO 家族原理

**配套讲义**: [`days/day-25.md`](../days/day-25.md) ｜ **需要 GPU（云机器）**

手写 DPO loss（含 reference 项）并用玩具数据验证数值 —— 重点是跑通那个 **`ln 2` 检查**：chosen 与 rejected 完全相同时 loss 必须等于 0.6931。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w5.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 手工推一遍公式，再跑验证

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.dpo_loss", "--verify"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 自己实现一遍 DPO loss（不看源码）

只有三行：
```python
chosen_r  = beta * (pi_chosen  - ref_chosen)
rejected_r = beta * (pi_rejected - ref_rejected)
loss = -F.logsigmoid(chosen_r - rejected_r).mean()
```

In [ ]:
import math

def my_dpo(pc, pr, rc, rr, beta=0.1):
    cr = beta * (pc - rc)
    rj = beta * (pr - rr)
    d = cr - rj
    return -math.log(1 / (1 + math.exp(-d)))     # -log sigmoid(d)

# 场景 1：chosen 明显更好
print("chosen 好 →", round(my_dpo(-1.0, -2.0, -1.0, -2.0), 4))
# 场景 2：完全相同 → 应该正好是 ln2
print("完全一样 →", round(my_dpo(-1.5, -1.5, -1.0, -1.0), 4), " (ln2 =", round(math.log(2), 4), ")")
# 思考：β 变大 10 倍，场景 1 的 loss 怎么变？为什么？
for beta in (0.01, 0.1, 1.0):
    print(f"  beta={beta:<5} loss={my_dpo(-1.0, -2.0, -1.0, -2.0, beta):.6f}")

## 3. 落笔：β 的直觉

β 大 → 保守（贴近 reference）／β 小 → 激进（易崩）。写下你的理解和一个具体的调参方案。

In [ ]:
beta_note = """
β 的物理意义：
客服场景我选 β = ___，理由是：
"""
print(beta_note)

## 验收清单

- [ ] `--verify` 四个场景断言全部通过，**特别是 ln2 = 0.6931 那一项**
- [ ] 能自己推出 DPO 的闭式解（不用翻论文）
- [ ] 能说清 β 的作用，以及为什么典型值是 0.1–0.5
- [ ] 知道 ORPO / SimPO / KTO 分别省掉了什么（reference / 配对假设 / 正负样本）

**卡住了？** 回看 [`days/day-25.md`](../days/day-25.md) 第五节「容易踩的坑」。

> **明天**：`days/day-26.md` —— 造偏好数据（本地可跑，不用开 GPU）